# Customer Churn Prediction and Retention Analytics System

This notebook walks through the full pipeline for the Telco-style customer
churn project: data cleaning, exploratory data analysis, feature engineering,
encoding, scaling, model training (Logistic Regression, Random Forest,
XGBoost), evaluation, and business recommendations.

**Note on the dataset:** this environment doesn't have internet access, so
`data/telco_customer_churn_raw.csv` is a synthetic dataset generated by
`src/generate_dataset.py` to match the schema and statistical patterns of the
real IBM Telco Customer Churn dataset (same columns, same kinds of
missing-data/duplicate issues, same churn drivers). **If you have the real
Kaggle CSV, just replace that file with it (same column names) and every cell
below works unchanged.**


In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_pipeline import load_data, clean_data, engineer_features, encode_features, scale_features

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

RAW_PATH = '../data/telco_customer_churn_raw.csv'


## 1. Load Raw Data & Assess Data Quality

In [ ]:
df_raw = load_data(RAW_PATH)
print(df_raw.shape)
df_raw.head()


In [ ]:
# Data quality assessment
print("Missing values per column:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])
print("\nDuplicate rows:", df_raw.duplicated().sum())
print("\nTotalCharges dtype (should reveal the ' 123.45 ' string issue):", df_raw['TotalCharges'].dtype)
print(df_raw['TotalCharges'].apply(lambda v: isinstance(v, str)).sum(), "rows stored TotalCharges as text")


### Data quality issues found
- **Missing values** in `TotalCharges` (mostly brand-new customers with 0 tenure), `MonthlyCharges`, `gender`, and `PaymentMethod`.
- **Duplicate rows** (25 injected duplicates).
- **Inconsistent formatting**: `gender` and `Churn` have mixed case and stray whitespace.
- **Type issues**: `TotalCharges` is partly stored as text with leading/trailing spaces.
- **Outliers**: a handful of `MonthlyCharges` values are implausibly high (data-entry spikes).

## 2. Data Cleaning

In [ ]:
df_clean = clean_data(df_raw)
print("Shape after cleaning:", df_clean.shape, "(started at", df_raw.shape, ")")
print("Rows dropped as exact duplicates:", df_clean.attrs.get('rows_dropped_as_duplicates'))
print("\nRemaining missing values:", df_clean.isna().sum().sum())
df_clean.head()


**Cleaning steps applied** (see `src/data_pipeline.py::clean_data`):
- Stripped whitespace and normalized case on text columns.
- Converted `TotalCharges` to numeric (coercing the stray text entries).
- Dropped exact duplicate rows.
- **Numerical imputation**: median for `MonthlyCharges` and `TotalCharges` (robust to outliers).
- **Categorical imputation**: mode for `gender` and `PaymentMethod`.
- Capped `MonthlyCharges` outliers at the 1.5×IQR upper bound.
- Removed any rows with invalid `tenure` or `Churn` values.

## 3. Exploratory Data Analysis

In [ ]:
churn_rate = (df_clean['Churn'] == 'Yes').mean()
print(f"Overall churn rate: {churn_rate:.1%}")

plt.figure(figsize=(5,4))
ax = sns.countplot(data=df_clean, x='Churn', hue='Churn', palette=['#3b6fb6','#c0533b'], legend=False)
ax.set_title('Churn Distribution')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col in zip(axes, ['tenure','MonthlyCharges','TotalCharges']):
    sns.histplot(df_clean[col], bins=30, kde=True, ax=ax, color='#3b6fb6')
    ax.set_title(f'Distribution of {col}')
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col in zip(axes, ['tenure','MonthlyCharges','TotalCharges']):
    sns.boxplot(data=df_clean, x='Churn', y=col, hue='Churn', ax=ax, palette=['#3b6fb6','#c0533b'], legend=False)
    ax.set_title(f'{col} by Churn')
plt.tight_layout(); plt.show()


In [ ]:
# Demographics analysis
fig, axes = plt.subplots(2, 2, figsize=(12,9))
for ax, col in zip(axes.flatten(), ['gender','SeniorCitizen','Partner','Dependents']):
    sns.countplot(data=df_clean, x=col, hue='Churn', ax=ax, palette=['#3b6fb6','#c0533b'])
    ax.set_title(f'Churn by {col}')
plt.tight_layout(); plt.show()


In [ ]:
# Contract & payment behavior analysis
fig, axes = plt.subplots(1, 2, figsize=(13,4.5))
sns.countplot(data=df_clean, x='Contract', hue='Churn', ax=axes[0], palette=['#3b6fb6','#c0533b'])
axes[0].set_title('Churn by Contract Type')
sns.countplot(data=df_clean, x='PaymentMethod', hue='Churn', ax=axes[1], palette=['#3b6fb6','#c0533b'])
axes[1].set_title('Churn by Payment Method')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

print(df_clean.groupby('Contract')['Churn'].apply(lambda s: (s=='Yes').mean()).sort_values(ascending=False))


In [ ]:
# Correlation heatmap (numeric features)
numeric_df = df_clean.copy()
numeric_df['Churn_flag'] = (numeric_df['Churn']=='Yes').astype(int)
numeric_df['SeniorCitizen'] = numeric_df['SeniorCitizen'].astype(int)
corr = numeric_df[['tenure','MonthlyCharges','TotalCharges','SeniorCitizen','Churn_flag']].corr()
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout(); plt.show()


### EDA Insights
- Churn is concentrated in **month-to-month contracts**, **fiber-optic internet**, and **electronic-check payers**.
- **Tenure** is strongly negatively associated with churn — new customers (low tenure) churn far more.
- `MonthlyCharges` is mildly positively correlated with churn; `TotalCharges` is negatively correlated (a proxy for tenure).

## 4. Feature Engineering

In [ ]:
df_fe = engineer_features(df_clean)
new_cols = ['TenureSegment','SpendTier','TotalRevenue','TotalRevenueLog','ServiceCount','ContractRiskScore','AvgMonthlySpend']
df_fe[new_cols].head()


**Engineered features:**
- `TenureSegment`: New (≤12mo) / Regular (13-48mo) / Loyal (>48mo)
- `SpendTier`: Low / Medium / High, based on `MonthlyCharges` terciles
- `TotalRevenue` / `TotalRevenueLog`: revenue feature with a log transform to reduce skew
- `ServiceCount`: number of subscribed add-on services
- `ContractRiskScore`: 3 = month-to-month (highest risk) down to 1 = two-year contract
- `AvgMonthlySpend`: total revenue divided by tenure (avg spend rate)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
sns.countplot(data=df_fe, x='TenureSegment', hue='Churn', ax=axes[0], palette=['#3b6fb6','#c0533b'],
              order=['New','Regular','Loyal'])
axes[0].set_title('Churn by Tenure Segment')
sns.countplot(data=df_fe, x='SpendTier', hue='Churn', ax=axes[1], palette=['#3b6fb6','#c0533b'],
              order=['Low','Medium','High'])
axes[1].set_title('Churn by Spend Tier')
sns.boxplot(data=df_fe, x='Churn', y='ServiceCount', hue='Churn', ax=axes[2], palette=['#3b6fb6','#c0533b'], legend=False)
axes[2].set_title('Service Count by Churn')
plt.tight_layout(); plt.show()


## 5. Categorical Encoding

In [ ]:
df_encoded, encoders = encode_features(df_fe, fit_encoders=True)
print("Shape after encoding:", df_encoded.shape)
df_encoded.head()


- **Label Encoding** for binary Yes/No columns: `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `Churn`.
- **One-Hot Encoding** for nominal columns: `InternetService`, `PaymentMethod`, `Contract`, `gender`, `TenureSegment`, `SpendTier` (with `drop_first=True` to avoid the dummy-variable trap).

## 6. Feature Scaling

In [ ]:
df_scaled, scaler = scale_features(df_encoded, fit_scaler=True)
print("Scaled columns preview:")
df_scaled[['tenure','MonthlyCharges','TotalCharges']].describe()


`StandardScaler` was used for `tenure`, `MonthlyCharges`, `TotalCharges`, and the engineered revenue features, since these are roughly continuous and Logistic Regression benefits from zero-mean/unit-variance inputs. (`MinMaxScaler` is also available in `src/data_pipeline.py` if a bounded [0,1] range is preferred for a different model.)

## 7. Model Development

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("xgboost not installed - falling back to GradientBoostingClassifier. "
          "`pip install xgboost` to use real XGBoost; no other code changes needed.")

X = df_scaled.drop(columns=['Churn'])
y = df_scaled['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5,
                                             class_weight='balanced', random_state=42, n_jobs=-1),
}
if HAS_XGBOOST:
    scale_pos_weight = (y_train==0).sum() / max((y_train==1).sum(),1)
    models['XGBoost'] = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                           subsample=0.8, colsample_bytree=0.8,
                                           scale_pos_weight=scale_pos_weight,
                                           eval_metric='logloss', random_state=42, n_jobs=-1)
else:
    models['XGBoost'] = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)

trained = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    trained[name] = m
print("Trained:", list(trained.keys()))


## 8. Model Evaluation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, m in trained.items():
    y_pred = m.predict(X_test)
    y_proba = m.predict_proba(X_test)[:,1]
    cv = cross_val_score(m, X_train, y_train, cv=skf, scoring='roc_auc')
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test,y_pred),
        'Precision': precision_score(y_test,y_pred),
        'Recall': recall_score(y_test,y_pred),
        'F1': f1_score(y_test,y_pred),
        'ROC-AUC': roc_auc_score(y_test,y_proba),
        'CV ROC-AUC': cv.mean(),
    })
results_df = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, name in zip(axes, trained.keys()):
    cm = confusion_matrix(y_test, trained[name].predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=['No Churn','Churn']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(6,5))
for name, m in trained.items():
    y_proba = m.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test,y_proba):.3f})")
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()


## 9. Feature Importance & Retention Insights

In [ ]:
rf = trained['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
plt.figure(figsize=(7,6))
importances.sort_values().plot(kind='barh', color='#3b6fb6')
plt.title('Top 15 Feature Importances - Random Forest')
plt.tight_layout(); plt.show()
importances


### Business Recommendations
1. **Target month-to-month customers early** — they churn at roughly 3-4x the rate of two-year contract customers. Offering a discounted 1-year upgrade in the first 3 months of tenure could meaningfully cut churn.
2. **Fiber-optic customers churn more than DSL** — investigate whether this is price-driven (fiber is the most expensive tier) or service-quality driven, and consider a loyalty discount for fiber subscribers past their 6-month mark.
3. **Electronic-check payers churn the most among payment methods** — prompting a switch to autopay (bank transfer or credit card) is correlated with materially lower churn.
4. **New customers (≤12 months tenure) are the highest-risk segment** — a structured onboarding/check-in program in months 1-3 targets the point of highest attrition risk.
5. **Add-on services (security, tech support) correlate with retention** — bundling a free trial of these services with new sign-ups may improve stickiness.

The trained models are saved to `models/` and served by the Streamlit dashboard in `app.py` for interactive churn prediction on new customers.